<a href="https://colab.research.google.com/github/Naufallm/-RAG-Quantitative-Research-Engine-for-IDX/blob/main/DSC_Project_IDX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Member
- Ahmad Naufal Luthfan Marzuqi
- Syahrial Nur Faturrahman

# SCRAPING → RAG


## Install library

Install library utama untuk:

- parsing PDF (pymupdf, pdfplumber)
- RAG (langchain)
- vector DB (faiss)
- embedding (openai)

In [1]:
!pip install -q \
langchain==0.2.14 \
langchain-community==0.2.12 \
langchain-text-splitters==0.2.2 \
faiss-cpu==1.7.4 \
pymupdf==1.24.1 \
pdfplumber==0.11.0 \
sentence-transformers==2.7.0 \
requests==2.32.4

ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2)
ERROR: No matching distribution found for faiss-cpu==1.7.4


In [ ]:
import os
os.kill(os.getpid(), 9)

## Mount Drive

Menghubungkan Colab ke Google Drive agar bisa akses dataset PDF

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/dataset_idx"

## Import Library

Import library untuk:

- membaca file
- parsing PDF

In [ ]:
import os
import fitz
import pdfplumber

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

## Function Parsing PDF

Fungsi ini bertujuan untuk
- Mengambil teks biasa (PyMuPDF)
- Mengambil tabel (pdfplumber)

Tujuannya untuk Menggabungkan semua informasi (text + tabel)

In [ ]:
def extract_text_from_pdf(pdf_path):
    text = ""

    try:
        # 🔹 PyMuPDF (text utama)
        doc = fitz.open(pdf_path)
        for page in doc:
            text += page.get_text()

        # 🔹 pdfplumber (tabel)
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                table = page.extract_table()
                if table:
                    for row in table:
                        row_text = " | ".join([str(cell) for cell in row])
                        text += row_text + "\n"

    except Exception as e:
        print(f"Error parsing {pdf_path}: {e}")

    return text

## Load Dataset

In [ ]:
all_texts = []
file_names = []

for file in os.listdir(DATASET_PATH):
    if file.endswith(".pdf"):
        path = os.path.join(DATASET_PATH, file)
        print("Processing:", file)

        text = extract_text_from_pdf(path)

        if len(text.strip()) > 0:
            all_texts.append(text)
            file_names.append(file)

print("Total dokumen berhasil diproses:", len(all_texts))

Processing: FinancialStatement-2026-I-AGRO.pdf
Processing: FinancialStatement-2026-I-AKRA.pdf
Processing: FinancialStatement-2026-I-ARTO.pdf
Processing: FinancialStatement-2026-I-AUTO.pdf
Processing: FinancialStatement-2026-I-BANK.pdf
Processing: FinancialStatement-2026-I-BBCA.pdf
Processing: FinancialStatement-2026-I-BBSS.pdf
Processing: FinancialStatement-2026-I-BBTN.pdf
Processing: FinancialStatement-2026-I-BEKS.pdf
Processing: FinancialStatement-2026-I-BINA.pdf
Processing: FinancialStatement-2026-I-BLOG.pdf
Processing: FinancialStatement-2026-I-BMRI.pdf
Processing: FinancialStatement-2026-I-BNLI.pdf
Processing: FinancialStatement-2026-I-BOBA.pdf
Processing: FinancialStatement-2026-I-BOLT.pdf
Processing: FinancialStatement-2026-I-CFIN.pdf
Processing: FinancialStatement-2026-I-DCII.pdf
Processing: FinancialStatement-2026-I-EAST.pdf
Processing: FinancialStatement-2026-I-EDGE.pdf
Processing: FinancialStatement-2026-I-FORE.pdf
Processing: FinancialStatement-2026-I-GIAA.pdf
Processing: F

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = []
metadata = []

for i, doc in enumerate(all_texts):
    split_texts = text_splitter.split_text(doc)

    for chunk in split_texts:
        chunks.append(chunk)
        metadata.append({"source": file_names[i]})

print("Total chunk:", len(chunks))
print("Contoh chunk:\n", chunks[0][:300])

Total chunk: 7546
Contoh chunk:
 Perseroan dengan ini menyampaikan laporan keuangan untuk periode 3 Bulan yang berakhir pada 31/03/2026 dengan ikhtisar sebagai berikut : 
  
Informasi mengenai anak perusahaan Perseroan sebagai berikut : 
  
  
  
  
Dokumen ini merupakan dokumen resmi PT Bank Raya Indonesia Tbk yang tidak memerluka


In [ ]:
import os
from langchain_openai import OpenAIEmbeddings

os.environ["OPENAI_API_KEY"] = "ISI_API_KEY_KAMU"

embeddings = OpenAIEmbeddings()

In [ ]:
from langchain.vectorstores import FAISS

vectorstore = FAISS.from_texts(chunks, embeddings, metadatas=metadata)

print("Vector DB berhasil dibuat!")

ModuleNotFoundError: No module named 'langchain.vectorstores'